In [22]:
import gymnasium as gym
import numpy as np
import tensorboard
import torch
import torch.nn as nn
import tempfile, shutil, os
from torch.utils.tensorboard import SummaryWriter

shutil.rmtree(os.path.join(tempfile.gettempdir(), ".tensorboard-info"), ignore_errors=True)

In [23]:
env = gym.make("FrozenLake-v1", is_slippery=True, render_mode = "ansi")
n_states = env.observation_space.n
n_actions = env.action_space.n

In [24]:
#Q-Learning Model
def train_q_learning(env, n_episodes=20000, alpha=0.1, gamma=0.99, 
                     epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.9995, writer=None):
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions))
    results = []

    for episode in range(n_episodes):
        state, info = env.reset()
        done = False
        while not done:
            if np.random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(Q[state])

            next_state, reward, terminated, truncated, info = env.step(action)

            if terminated:
                target = reward
            else:
                target = reward + gamma * np.max(Q[next_state])

            Q[state, action] = Q[state, action] + alpha * (target - Q[state, action])

            done = terminated or truncated
            state = next_state

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        results.append(reward)
        if writer:
            writer.add_scalar("success_rate", np.mean(results[-100:]), episode)
            writer.add_scalar("epsilon", epsilon, episode)

    return Q

In [25]:
#REINFORCE Model
class PolicyNet(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 64)
        self.fc2 = nn.Linear(64, n_out)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

policy = PolicyNet(16,4)

def run_episode(env, policy):
    log_probs = []
    rewards = []
    state, info = env.reset()
    done = False

    while not done:
        one_hot_vector = torch.nn.functional.one_hot(torch.tensor(state), 16).float()
        scores = policy(one_hot_vector)
        dist = torch.distributions.Categorical(logits=scores)
        action = dist.sample()
        log_probs.append(dist.log_prob(action))
        next_state, reward, terminated, truncated, info = env.step(action.item())
        rewards.append(reward)
        done = terminated or truncated
        state = next_state
        
    return log_probs, rewards

def compute_returns(rewards, gamma = 0.99):
    G = 0
    returns = []

    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.append(G)

    returns.reverse()
    return torch.tensor(returns)

def train_reinforce(env, policy, optimizer, n_episodes = 5000, gamma = 0.99, writer = None):
    results = []
    for episode in range (n_episodes):
        log_probs, rewards = run_episode(env, policy)
        returns = compute_returns(rewards, gamma)
        loss = -(torch.stack(log_probs) * returns).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        results.append(rewards[-1])
        if writer:
            writer.add_scalar("success_rate", np.mean(results[-100:]), episode)

    return policy
        
def train_reinforce_batch(env, policy, optimizer, n_updates=2000, batch_size=16, gamma=0.99, writer=None):
    results = []
    for update in range(n_updates):
        all_log_probs, all_returns = [], []
        for _ in range(batch_size):
            log_probs, rewards = run_episode(env, policy)
            all_log_probs.append(torch.stack(log_probs))
            all_returns.append(compute_returns(rewards, gamma))
            results.append(rewards[-1])

        log_probs = torch.cat(all_log_probs)
        returns = torch.cat(all_returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        loss = -(log_probs * returns).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if writer:
            writer.add_scalar("success_rate", np.mean(results[-100:]), update * batch_size)
    return policy

In [26]:
#PPO Model
class ActorCritic(nn.Module):
    def __init__(self, n_in, n_actions):
        super().__init__()
        self.actor = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU(), nn.Linear(64, n_actions))
        self.critic = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x):
        return self.actor(x), self.critic(x)

def collect_episode(env, model):
    states, actions, log_probs, values, rewards = [], [], [], [], []
    state, info = env.reset()
    done = False
    while not done:
        x = torch.nn.functional.one_hot(torch.tensor(state), 16).float()
        with torch.no_grad():
            scores, value = model(x)
        dist = torch.distributions.Categorical(logits=scores)
        action = dist.sample()

        states.append(x)
        actions.append(action)
        log_probs.append(dist.log_prob(action))
        values.append(value.squeeze())

        state, reward, terminated, truncated, info = env.step(action.item())
        rewards.append(reward)
        done = terminated or truncated

    return torch.stack(states), torch.stack(actions), torch.stack(log_probs), torch.stack(values), rewards

def compute_advantages(rewards, values, gamma = 0.99):
    returns = compute_returns(rewards, gamma)
    advantages = returns - values
    return returns, advantages

def collect_batch(env, model, batch_size = 16, gamma = 0.99):
    all_s, all_a, all_lp, all_ret, all_adv = [], [], [], [], []
    results = []
    for _ in range(batch_size):
        s, a, lp, v, r = collect_episode(env, model)
        returns, adv = compute_advantages(r, v, gamma)
        all_s.append(s)
        all_a.append(a)
        all_lp.append(lp)
        all_ret.append(returns)
        all_adv.append(adv)
        results.append(r[-1])

    adv = torch.cat(all_adv)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    return torch.cat(all_s), torch.cat(all_a), torch.cat(all_lp), torch.cat(all_ret), adv, results

def ppo_loss(model, s, a, old_lp, ret, adv, clip = 0.2, vf_coef = 0.5, ent_coef = 0.01):
    scores, values = model(s)
    dist = torch.distributions.Categorical(logits=scores)
    new_lp = dist.log_prob(a)

    ratio = torch.exp(new_lp - old_lp)
    unclipped = ratio * adv
    clipped = torch.clamp(ratio, 1 - clip, 1 + clip) * adv
    actor_loss = -torch.min(unclipped, clipped).mean()

    critic_loss = ((values.squeeze(-1) - ret) ** 2).mean()
    entropy = dist.entropy().mean()
    return actor_loss + vf_coef * critic_loss - ent_coef * entropy
    

def train_ppo(env, model, optimizer, n_updates = 1000, batch_size = 16, n_epochs = 4, gamma = 0.99, clip = 0.2, writer = None):
    results = []
    for update in range(n_updates):
        s, a, old_lp, ret, adv, batch_results = collect_batch(env, model, batch_size, gamma)
        results.extend(batch_results)

        for _ in range(n_epochs):
            loss = ppo_loss(model, s, a, old_lp, ret, adv, clip)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if writer:
            writer.add_scalar("success_rate", np.mean(results[-100:]), update * batch_size)
    return model

def greedy_action_ppo(model, state):
    with torch.no_grad():
        x = torch.nn.functional.one_hot(torch.tensor(state), 16).float()
        scores, _ = model(x)
        return torch.argmax(scores).item()


In [27]:
def greedy_action(policy, state):
    with torch.no_grad():
        x = torch.nn.functional.one_hot(torch.tensor(state), 16).float()
        return torch.argmax(policy(x)).item()

In [28]:
def evaluate_policy(env, choose_action, n_episodes=200):
    successes = 0
    for episode in range(n_episodes):
        state, info = env.reset()
        done = False
        while not done:
            action = choose_action(state)
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
        successes += reward
    return successes / n_episodes

In [29]:
shutil.rmtree("runs/frozenlake/q_learning", ignore_errors=True)
writer = SummaryWriter("runs/frozenlake/q_learning7")

Q = train_q_learning(env, writer=writer)
writer.close()

print(evaluate_policy(env, lambda s: np.argmax(Q[s])))
print(np.argmax(Q, axis=1).reshape(4, 4))

0.715
[[0 3 3 3]
 [0 0 0 0]
 [3 1 0 0]
 [0 2 1 0]]


In [30]:
shutil.rmtree("runs/frozenlake/reinforce_batch2", ignore_errors=True)
writer = SummaryWriter("runs/frozenlake/reinforce_batch8")

policy = PolicyNet(16, 4)
optimizer = torch.optim.Adam(policy.parameters(), lr=0.01)
train_reinforce_batch(env, policy, optimizer, n_updates=5000, batch_size=16, writer=writer)
writer.close()

print(evaluate_policy(env, lambda s: greedy_action(policy, s)))

0.1


In [31]:
shutil.rmtree("runs/frozenlake/ppo4", ignore_errors=True)
writer = SummaryWriter("runs/frozenlake/ppo6")

model = ActorCritic(16, 4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
train_ppo(env, model, optimizer, n_updates=500, batch_size=64, writer=writer)
writer.close()

print(evaluate_policy(env, lambda s: greedy_action_ppo(model, s), n_episodes=1000))

0.709


In [32]:
%load_ext tensorboard
%tensorboard --logdir runs
%reload_ext tensorboard